# Infold Stack Pipeline with XGBoost

In [1]:
import gc
import numpy as np
import pandas as pd
import torch
import os
from pathlib import Path
from time import time
from itertools import combinations

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from scipy.optimize import minimize
from scipy.stats import entropy

from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Configuration
SEED = 42
N_FOLDS = 5
N_SEEDS = 3
DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e4/')
WORKING_DIR = Path('/kaggle/working/')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'

USE_GPU = torch.cuda.is_available()

def seed_everything(seed=SEED):
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

In [3]:
# Feature expansion
# Feature Engineering (Inspired by waterworld)
top_cat_cols = ['Crop_Growth_Stage', 'Mulching_Used', 'Crop_Type']
top_num_cols = ['Soil_Moisture', 'Temperature_C', 'Wind_Speed_kmh', 'Rainfall_mm']
top_cols     = ['Soil_Moisture', 'Crop_Growth_Stage', 'Temperature_C',
                'Mulching_Used', 'Wind_Speed_kmh', 'Rainfall_mm']

In [4]:
# WARMUP CELL: Validates GPU configuration and parameters with dummy data before full execution.
def warmup_check():
    print('Running Warmup Check...')
    if not USE_GPU:
        print('WARNING: GPU is not detected! Training will be extremely slow. Please enable GPU in Kaggle settings.')
    else:
        print('SUCCESS: GPU Detected!')
    
    try:
        print('Validating XGBoost params...')
        XGBClassifier(objective='multi:softprob', num_class=3, tree_method='hist', device='cuda' if USE_GPU else 'cpu', early_stopping_rounds=1).fit(np.random.rand(100, 5), np.random.randint(0, 3, 100), eval_set=[(np.random.rand(20, 5), np.random.randint(0, 3, 20))], verbose=False)
        print('Validating LightGBM params...')
        lgb.LGBMClassifier(objective='multiclass', num_class=3, device='gpu' if USE_GPU else 'cpu').fit(np.random.rand(100, 5), np.random.randint(0, 3, 100), eval_set=[(np.random.rand(20, 5), np.random.randint(0, 3, 20))], callbacks=[lgb.early_stopping(1, verbose=False)])
        print('Validating CatBoost params...')
        CatBoostClassifier(iterations=2, loss_function='MultiClass', classes_count=3, task_type='GPU' if USE_GPU else 'CPU', verbose=0).fit(np.random.rand(100, 5), np.random.randint(0, 3, 100))
        print('\n--- WARMUP SUCCESSFUL! No parameter or GPU issues detected. You are safe to run the pipeline! ---')
    except Exception as e:
        print(f'\n!!! WARMUP FAILED: {str(e)} !!!')
        print('Please fix the configuration before running the full pipeline.')
        raise e

warmup_check()


Running Warmup Check...
SUCCESS: GPU Detected!
Validating XGBoost params...
Validating LightGBM params...
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 175
[LightGBM] [Info] Number of data points in the train set: 100, number of used features: 5
[LightGBM] [Info] Using GPU Device: Tesla P100-PCIE-16GB, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 64 bins...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 5 dense feature groups (0.00 MB) transferred to GPU in 0.000265 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score -0.941609
[LightGBM] [Info] Start training from score -1.049822
[LightGBM] [Info] Start training from score -1.347074
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

In [5]:
def add_threshold_and_domain_features(df):
    df['soil_lt_25']  = (df['Soil_Moisture']  < 25).astype(int)
    df['wind_gt_10']  = (df['Wind_Speed_kmh'] > 10).astype(int)
    df['temp_gt_30']  = (df['Temperature_C']  > 30).astype(int)
    df['rain_lt_300'] = (df['Rainfall_mm']    < 300).astype(int)
    df['moist_rain']    = df['Soil_Moisture'] / (df['Rainfall_mm'] + 1)
    df['moist_temp']    = df['Soil_Moisture'] / (df['Temperature_C'] + 1)
    df['moist_wind']    = df['Soil_Moisture'] / (df['Wind_Speed_kmh'] + 1)
    df['ET_proxy']      = (df['Temperature_C'] * df['Wind_Speed_kmh'] * df['Sunlight_Hours']) / (df['Humidity'] + 1)
    df['heat_stress']   = df['Temperature_C'] * df['Sunlight_Hours']
    df['drying_force']  = df['Wind_Speed_kmh'] * df['Temperature_C'] / (df['Humidity'] + 1)
    df['water_supply']  = df['Rainfall_mm'] + df['Previous_Irrigation_mm']
    df['water_deficit'] = df['Soil_Moisture'] - df['water_supply'] * 0.1
    df['soil_quality']  = df['Organic_Carbon'] / (df['Electrical_Conductivity'] + 0.1)
    df['moist_x_temp']  = df['Soil_Moisture'] * df['Temperature_C']
    df['wind_x_temp']   = df['Wind_Speed_kmh'] * df['Temperature_C']
    return df

def add_formula_score(df):
    df['high_score'] = ((df['Soil_Moisture'] < 25) * 2 +
                        (df['Rainfall_mm'] < 300) * 2 +
                        (df['Temperature_C'] > 30) * 1 +
                        (df['Wind_Speed_kmh'] > 10) * 1)
    df['low_score']  = ((df['Crop_Growth_Stage'].isin(['Harvest', 'Sowing'])) * 2 +
                        (df['Mulching_Used'] == 'Yes') * 1)
    df['formula_score'] = df['high_score'] - df['low_score']
    df['formula_pred']  = 0
    df.loc[(df['formula_score'] > 0) & (df['formula_score'] <= 3), 'formula_pred'] = 1
    df.loc[df['formula_score'] > 3, 'formula_pred'] = 2
    return df

In [6]:
def _add_ngram_categoricals(train_fe, test_fe, top_cat_cols):
    """Create bigram and trigram categorical interaction columns."""
    ngram_cols = []

    for c1, c2 in combinations(top_cat_cols, 2):
        name = f'BG_{c1}_{c2}'
        for df in (train_fe, test_fe):
            df[name] = df[c1].astype(str) + '_' + df[c2].astype(str)
        ngram_cols.append(name)

    for c1, c2, c3 in combinations(top_cat_cols, 3):
        name = f'TG_{c1}_{c2}_{c3}'
        for df in (train_fe, test_fe):
            df[name] = df[c1].astype(str) + '_' + df[c2].astype(str) + '_' + df[c3].astype(str)
        ngram_cols.append(name)

    for col in ngram_cols:
        combined = pd.concat([train_fe[col], test_fe[col]], axis=0).astype(str)
        labels, _ = pd.factorize(combined)
        train_fe[col] = labels[:len(train_fe)]
        test_fe[col]  = labels[len(train_fe):]

    return ngram_cols

In [7]:
def _add_bin_cat_interactions(train_fe, test_fe, top_num_cols, top_cat_cols):
    """Create binned numerical × categorical interaction columns."""
    bin_cat_int_cols = []

    for num_col in top_num_cols:
        bin_col = f'{num_col}_bin'
        train_fe[bin_col], bins = pd.qcut(
            train_fe[num_col], q=5, labels=False, retbins=True, duplicates='drop'
        )
        test_fe[bin_col] = (
            pd.cut(test_fe[num_col], bins=bins, labels=False, include_lowest=True)
            .fillna(0).astype(int)
        )

        for cat_col in top_cat_cols:
            name = f'{num_col}_bin_{cat_col}_int'
            for df in (train_fe, test_fe):
                df[name] = df[bin_col].astype(str) + '_' + df[cat_col].astype(str)
            bin_cat_int_cols.append(name)

    for df in (train_fe, test_fe):
        df.drop(columns=[f'{c}_bin' for c in top_num_cols], inplace=True)

    for col in bin_cat_int_cols:
        codes, uniques = pd.factorize(train_fe[col])
        train_fe[col] = codes.astype(int)
        mapping = {v: i for i, v in enumerate(uniques)}
        test_fe[col] = test_fe[col].map(mapping).fillna(-1).astype(int)

    return bin_cat_int_cols

In [8]:
def _add_group_aggregates(train_fe, test_fe, top_cat_cols, top_num_cols):
    """Add per-category group mean, diff, and ratio features."""
    num_cat_agg_cols = set()

    for cat_col in top_cat_cols:
        for num_col in top_num_cols:
            group_means = train_fe.groupby(cat_col)[num_col].mean()
            avg   = f'{num_col}_avg_by_{cat_col}'
            diff  = f'{num_col}_diff_{cat_col}'
            ratio = f'{num_col}_ratio_{cat_col}'
            for df in (train_fe, test_fe):
                df[avg]   = df[cat_col].map(group_means).astype(float)
                df[diff]  = (df[num_col] - df[avg]).astype(float)
                df[ratio] = (df[num_col] / (df[avg] + 1e-6)).astype(float)
            num_cat_agg_cols.update([avg, diff, ratio])

    return list(num_cat_agg_cols)

In [9]:
def _add_round_decimal_bin_features(train_fe, test_fe, top_num_cols):
    """Add rounding, decimal, and quantile-bin features."""
    round_cols, bin_cols = [], []

    round_config = {
        'Soil_Moisture':  [0, -1],
        'Temperature_C':  [-1],
        'Rainfall_mm':    [0, -1, -2, -3],
        'Wind_Speed_kmh': [0, -1],
    }
    for col, rs in round_config.items():
        for r in rs:
            feat = f'{col}_r{r}'
            for df in (train_fe, test_fe):
                df[feat] = df[col].round(r)
            round_cols.append(feat)

    for col in top_num_cols:
        feat = f'{col}_decimal'
        for df in (train_fe, test_fe):
            df[feat] = (df[col] % 1).round(2)
        round_cols.append(feat)

    for col in top_num_cols:
        feat = f'{col}_qbin'
        train_fe[feat], edges = pd.qcut(
            train_fe[col], q=10, labels=False, duplicates='drop', retbins=True
        )
        test_fe[col] = test_fe[col].clip(edges[0], edges[-1])
        test_fe[feat] = pd.cut(
            test_fe[col], bins=edges, labels=False, include_lowest=True
        ).astype(int)
        bin_cols.append(feat)

    return round_cols + bin_cols

In [10]:
def _add_pairwise_features(train_fe, test_fe, top_cols):
    """Create factorized pairwise interaction columns from all top columns."""
    pair_cols = []

    for c1, c2 in combinations(top_cols, 2):
        name = f'{c1}__{c2}'
        combined = pd.concat([
            train_fe[c1].astype(str) + '_' + train_fe[c2].astype(str),
            test_fe[c1].astype(str)  + '_' + test_fe[c2].astype(str),
        ], ignore_index=True)
        codes, _ = pd.factorize(combined)
        train_fe[name] = codes[:len(train_fe)]
        test_fe[name]  = codes[len(train_fe):]
        pair_cols.append(name)

    return pair_cols

In [11]:
def engineer_all(train_df, test_df):
    train_fe = train_df.copy()
    test_fe  = test_df.copy()

    for df in (train_fe, test_fe):
        add_threshold_and_domain_features(df)
        add_formula_score(df)

    ngram_cols       = _add_ngram_categoricals(train_fe, test_fe, top_cat_cols)
    bin_cat_int_cols = _add_bin_cat_interactions(train_fe, test_fe, top_num_cols, top_cat_cols)
    num_cat_agg_cols = _add_group_aggregates(train_fe, test_fe, top_cat_cols, top_num_cols)
    _add_round_decimal_bin_features(train_fe, test_fe, top_num_cols)
    pair_cols        = _add_pairwise_features(train_fe, test_fe, top_cols)

    extra_cols = pair_cols + num_cat_agg_cols + bin_cat_int_cols
    return train_fe, test_fe, extra_cols

In [12]:
seed_everything(SEED)
print("Loading data...")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

Loading data...


In [13]:
target_map = {'Low': 0, 'Medium': 1, 'High': 2}
train['Irrigation_Need'] = train['Irrigation_Need'].map(target_map)
y_train = train['Irrigation_Need'].values

test_ids = test['id'].copy()

print("Applying Mass Feature Engineering...")
X_train_raw = train.drop(columns=['id', 'Irrigation_Need'])
X_test_raw = test.drop(columns=['id'])

X_train, X_test, DROP_AFTER_TE = engineer_all(X_train_raw, X_test_raw)

cat_cols = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
            'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
for c in cat_cols:
    X_train[c] = X_train[c].astype('category')
    X_test[c] = X_test[c].astype('category')

TE_FEATURES = X_train.columns.tolist()
print(f"Total Features Before TE: {X_train.shape[1]}")

Applying Mass Feature Engineering...
Total Features Before TE: 122


In [14]:
# Models
params_xgb = {
    'objective': 'multi:softprob', 'num_class': 3,
    'n_estimators': 2600, 'learning_rate': 0.05,
    'max_depth': 3, 'subsample': 0.9, 'colsample_bytree': 0.8,
    'max_bin': 1100, 'eval_metric': 'mlogloss', 'tree_method': 'hist',
    'device': 'cuda' if USE_GPU else 'cpu',
    'enable_categorical': True, 'n_jobs': -1, 'verbosity': 0,
    'early_stopping_rounds': 150
}

params_lgb = {
    'objective': 'multiclass', 'num_class': 3,
    'n_estimators': 2600, 'learning_rate': 0.05,
    'num_leaves': 15, 'min_child_samples': 30,
    'subsample': 0.9, 'subsample_freq': 1, 'colsample_bytree': 0.8,
    'reg_alpha': 0.3, 'reg_lambda': 0.3, 'verbosity': -1, 'n_jobs': -1,
}
if USE_GPU: params_lgb['device'] = 'gpu'

params_cb = {
    'iterations': 2600, 'learning_rate': 0.05,
    'depth': 4, 'l2_leaf_reg': 3.0,
    'subsample': 0.9, 'bootstrap_type': 'Bernoulli',
    'loss_function': 'MultiClass', 'classes_count': 3,
    'verbose': 0, 'early_stopping_rounds': 150
}
if USE_GPU: params_cb['task_type'] = 'GPU'

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
MODELS = ['xgb', 'lgb', 'cb']
oof_proba  = {m: np.zeros((len(X_train), 3)) for m in MODELS}
test_proba = {m: np.zeros((len(X_test), 3)) for m in MODELS}

In [15]:
# MAIN LEVEL 1 Loop
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\\n--- Fold {fold+1} ---")
    X_tr = X_train.iloc[tr_idx].copy()
    y_tr = y_train[tr_idx]
    X_val = X_train.iloc[va_idx].copy()
    y_val = y_train[va_idx]
    X_te = X_test.copy()

    # In-fold multiclass TE on EVERY feature
    encoder = TargetEncoder(target_type='multiclass', smooth='auto', cv=5, random_state=SEED)
    X_tr_te = encoder.fit_transform(X_tr[TE_FEATURES], y_tr)
    X_val_te = encoder.transform(X_val[TE_FEATURES])
    X_te_te = encoder.transform(X_te[TE_FEATURES])

    te_cols = [f'TE_{c}_class{k}' for c in TE_FEATURES for k in range(3)]
    X_tr[te_cols] = X_tr_te
    X_val[te_cols] = X_val_te
    X_te[te_cols] = X_te_te

    # Drop high card features
    X_tr_final = X_tr.drop(columns=DROP_AFTER_TE)
    X_val_final = X_val.drop(columns=DROP_AFTER_TE)
    X_te_final = X_te.drop(columns=DROP_AFTER_TE)

    sw = compute_sample_weight('balanced', y_tr)

    # XGBoost
    seed_val_xgb = np.zeros((len(X_val), 3))
    seed_test_xgb = np.zeros((len(X_test), 3))
    for s in range(N_SEEDS):
        m = XGBClassifier(**{**params_xgb, 'random_state': SEED + s})
        m.fit(X_tr_final, y_tr, eval_set=[(X_val_final, y_val)], sample_weight=sw, verbose=False)
        seed_val_xgb += m.predict_proba(X_val_final) / N_SEEDS
        seed_test_xgb += m.predict_proba(X_te_final) / N_SEEDS
    oof_proba['xgb'][va_idx] = seed_val_xgb
    test_proba['xgb'] += seed_test_xgb / N_FOLDS
    print(f"  xgb Fold {fold+1} BA: {balanced_accuracy_score(y_val, seed_val_xgb.argmax(axis=1)):.5f}")

    # Light Gradient Boost
    X_tr_lgb, X_val_lgb, X_te_lgb = X_tr_final.copy(), X_val_final.copy(), X_te_final.copy()
    for c in cat_cols:
        if c in X_tr_lgb.columns:
            X_tr_lgb[c] = X_tr_lgb[c].cat.codes.astype(int)
            X_val_lgb[c] = X_val_lgb[c].cat.codes.astype(int)
            X_te_lgb[c] = X_te_lgb[c].cat.codes.astype(int)
    
    seed_val_lgb = np.zeros((len(X_val), 3))
    seed_test_lgb = np.zeros((len(X_test), 3))
    for s in range(N_SEEDS):
        m = lgb.LGBMClassifier(**{**params_lgb, 'random_state': SEED + s})
        m.fit(X_tr_lgb, y_tr, sample_weight=sw, eval_set=[(X_val_lgb, y_val)], callbacks=[lgb.early_stopping(150, verbose=False)])
        seed_val_lgb += m.predict_proba(X_val_lgb) / N_SEEDS
        seed_test_lgb += m.predict_proba(X_te_lgb) / N_SEEDS
    oof_proba['lgb'][va_idx] = seed_val_lgb
    test_proba['lgb'] += seed_test_lgb / N_FOLDS
    print(f"  lgb Fold {fold+1} BA: {balanced_accuracy_score(y_val, seed_val_lgb.argmax(axis=1)):.5f}")

    # Cat Boost
    cat_features_cb = [c for c in cat_cols if c in X_tr_final.columns]
    X_tr_cb, X_val_cb, X_te_cb = X_tr_final.copy(), X_val_final.copy(), X_te_final.copy()
    for c in cat_features_cb:
        X_tr_cb[c], X_val_cb[c], X_te_cb[c] = X_tr_cb[c].astype(str), X_val_cb[c].astype(str), X_te_cb[c].astype(str)
    
    seed_val_cb = np.zeros((len(X_val), 3))
    seed_test_cb = np.zeros((len(X_test), 3))
    for s in range(N_SEEDS):
        m = CatBoostClassifier(**{**params_cb, 'random_seed': SEED + s})
        m.fit(X_tr_cb, y_tr, sample_weight=sw, eval_set=(X_val_cb, y_val), cat_features=cat_features_cb, verbose=False)
        seed_val_cb += m.predict_proba(X_val_cb) / N_SEEDS
        seed_test_cb += m.predict_proba(X_te_cb) / N_SEEDS
    oof_proba['cb'][va_idx] = seed_val_cb
    test_proba['cb'] += seed_test_cb / N_FOLDS
    print(f"  cb Fold {fold+1} BA: {balanced_accuracy_score(y_val, seed_val_cb.argmax(axis=1)):.5f}")


\n--- Fold 1 ---
  xgb Fold 1 BA: 0.97673


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


  lgb Fold 1 BA: 0.97603
  cb Fold 1 BA: 0.97737
\n--- Fold 2 ---
  xgb Fold 2 BA: 0.97831
  lgb Fold 2 BA: 0.97624
  cb Fold 2 BA: 0.97880
\n--- Fold 3 ---
  xgb Fold 3 BA: 0.97894
  lgb Fold 3 BA: 0.97797
  cb Fold 3 BA: 0.97998
\n--- Fold 4 ---
  xgb Fold 4 BA: 0.97799
  lgb Fold 4 BA: 0.97577
  cb Fold 4 BA: 0.97857
\n--- Fold 5 ---
  xgb Fold 5 BA: 0.97873
  lgb Fold 5 BA: 0.97627
  cb Fold 5 BA: 0.97958


In [16]:
# Meta Features
def compute_meta_features(model_probas):
    stacked = np.stack([model_probas[m] for m in model_probas], axis=0)
    mean_proba = stacked.mean(axis=0)
    std_proba = stacked.std(axis=0)
    feats = pd.DataFrame()
    for c, name in enumerate(['Low', 'Medium', 'High']):
        feats[f'mean_{name}'] = mean_proba[:, c]
        feats[f'std_{name}'] = std_proba[:, c]
        feats[f'var_{name}'] = std_proba[:, c] ** 2
    sorted_mean = np.sort(mean_proba, axis=1)
    feats['margin'] = sorted_mean[:, -1] - sorted_mean[:, -2]
    feats['max_prob'] = sorted_mean[:, -1]
    feats['entropy'] = -np.sum(mean_proba * np.log(mean_proba + 1e-9), axis=1)
    argmaxes = stacked.argmax(axis=2)
    majority = mean_proba.argmax(axis=1)
    feats['agreement'] = (argmaxes == majority[None, :]).mean(axis=0)
    return feats.astype(np.float32)

In [17]:
# LEVEL 2 Stack training the meta learner and threshold
meta_oof = compute_meta_features(oof_proba)
meta_test = compute_meta_features(test_proba)

X_meta_oof = np.hstack([np.hstack([oof_proba[m] for m in MODELS]), meta_oof.values])
X_meta_test = np.hstack([np.hstack([test_proba[m] for m in MODELS]), meta_test.values])

# Stack
skf_meta = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED+999)
stack_oof = np.zeros((len(X_meta_oof), 3))
stack_test = np.zeros((len(X_meta_test), 3))
for tr_idx, va_idx in skf_meta.split(X_meta_oof, y_train):
    meta = LogisticRegression(C=1.0, max_iter=500, n_jobs=-1)
    sw_meta = compute_sample_weight('balanced', y_train[tr_idx])
    meta.fit(X_meta_oof[tr_idx], y_train[tr_idx], sample_weight=sw_meta)
    stack_oof[va_idx] = meta.predict_proba(X_meta_oof[va_idx])
    stack_test += meta.predict_proba(X_meta_test) / N_FOLDS

stack_ba = balanced_accuracy_score(y_train, stack_oof.argmax(axis=1))
print(f'\\nStacked OOF BA (Before Tuning): {stack_ba:.5f}')

\nStacked OOF BA (Before Tuning): 0.97957


In [18]:
# Nelder-Mead Post-hoc Weight Tuning
def neg_ba(weights, proba, y_true):
    return -balanced_accuracy_score(y_true, (proba * weights).argmax(axis=1))

print("\\nTuning class weights...")
BOUNDS = [(0.7, 1.5)] * 3
TUNING_SEEDS = [0, 1, 2, 3, 4]

all_w = []
for seed in TUNING_SEEDS:
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y_train)); rng.shuffle(idx)
    half = len(idx) // 2
    f1, f2 = idx[:half], idx[half:]
    seed_w = []
    for tune_idx, _ in [(f1, f2), (f2, f1)]:
        best = (None, -np.inf)
        for x0 in [[1,1,1], [1.2,0.9,1], [1.4,0.85,1.05]]:
            res = minimize(neg_ba, x0=x0, args=(stack_oof[tune_idx], y_train[tune_idx]),
                           method='Nelder-Mead', bounds=BOUNDS, options={'xatol': 1e-4, 'maxiter': 300})
            if -res.fun > best[1]:
                best = (res.x, -res.fun)
        seed_w.append(best[0])
    all_w.append(np.mean(seed_w, axis=0))
    
stack_w = np.mean(all_w, axis=0)
final_ba = balanced_accuracy_score(y_train, (stack_oof * stack_w).argmax(axis=1))
print(f'Final Stack Tuned OOF BA: {final_ba:.5f} (Averaged weights: {stack_w.round(4)})')

\nTuning class weights...
Final Stack Tuned OOF BA: 0.97964 (Averaged weights: [1.199  0.9091 1.0408])


In [19]:
# Submissions
reverse_map = {0: 'Low', 1: 'Medium', 2: 'High'}
def make_sub(proba, name):
    pred = proba.argmax(axis=1)
    sub = pd.DataFrame({'id': test_ids, 'Irrigation_Need': [reverse_map[p] for p in pred]})
    sub.to_csv(WORKING_DIR / f'{name}.csv', index=False)

make_sub(stack_test * stack_w, 'submission')
print("Done! Submission saved as submission.csv")

Done! Submission saved as submission.csv
